# Etapa 2 - Pré-processamento da base NSL-KDD

**Entrega de 07/11 - Grupo 5**

Este notebook implementa e compara as decisões obrigatórias de pré-processamento. O classificador de referência serve apenas para medir o efeito das estratégias; a comparação definitiva de classificadores pertence à Etapa 3.

## 1. Decisões herdadas da Etapa 1

A análise exploratória encontrou 69.833 registros, 50 colunas, 94,16% de registros normais e 5,84% de anomalias. As regras abaixo são aplicadas de forma auditável:

- remoção de duplicatas exatas, preservando a primeira ocorrência;
- remoção de `num_outbound_cmds`, por ser constante;
- remoção de `src_bytes_kb`, `dst_bytes_kb` e `total_bytes`, mantendo as representações originais em bytes;
- exclusão preventiva de `tipo_alerta_soc`, `score_ids_legado` e `tempo_resposta_analista_min`, devido ao risco de informação operacional ou posterior ao evento;
- normalização de caixa e espaços em `protocol_type`, `service` e `flag`;
- valores impossíveis em taxas, contagens e grandezas não negativas são convertidos em ausentes, sem inventar fatores de escala;
- grupos com preditores idênticos permanecem na mesma dobra de validação.

Medianas, modas, categorias e parâmetros de escala são aprendidos somente no treino de cada dobra.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing_stage2 import (
    evaluate_cleaning_strategies,
    evaluate_preprocessing_strategies,
    prepare_modeling_data,
    report_as_dict,
)

pd.set_option('display.max_columns', 100)

## 2. Carregamento

O código procura primeiro a base usada na Etapa 1 e depois o arquivo oficial de treino. O conjunto de teste sem rótulo não é usado nos experimentos.

In [ ]:
data_candidates = [
    PROJECT_ROOT / 'base_nsl_kdd.csv',
    PROJECT_ROOT / 'data' / 'base_nsl_kdd.csv',
    PROJECT_ROOT / 'data' / 'treino.csv',
]
DATA_PATH = next((path for path in data_candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        'Base não encontrada. Adicione base_nsl_kdd.csv na raiz ou treino.csv em data/.'
    )

raw_data = pd.read_csv(DATA_PATH, sep=None, engine='python')
print('Arquivo:', DATA_PATH)
print('Dimensões brutas:', raw_data.shape)
display(raw_data.head())

## 3. Comparação de redundâncias e inconsistências

As variantes abaixo alteram uma decisão de limpeza por vez: normalização categórica e remoção de redundâncias, recuperação determinística de bytes ausentes, valores inválidos mantidos versus convertidos em ausentes versus limitados, remoção de duplicatas e tratamento conservador dos conflitos. A limpeza de linhas ocorre apenas no treino de cada dobra.

In [ ]:
cleaning_results = evaluate_cleaning_strategies(raw_data, n_splits=5)
display(cleaning_results.style.format({column: '{:.4f}' for column in cleaning_results.columns if column.endswith(('_media', '_desvio'))}))

## 4. Tratamento determinístico escolhido

Essas regras foram definidas antes da validação e não estimam parâmetros a partir das dobras. Valores inválidos viram `NaN`; a imputação propriamente dita continua dentro do pipeline. Rótulos conflitantes não são alterados automaticamente.

In [ ]:
X, y, preparation_report = prepare_modeling_data(raw_data)
display(pd.Series(report_as_dict(preparation_report), name='resultado').to_frame())
display(y.value_counts().rename('quantidade').to_frame())
print('Dimensões após regras determinísticas:', X.shape)

## 5. Técnicas comparadas

### Codificação simbólico-numérica

Utilizamos `OneHotEncoder(handle_unknown='ignore')`. A codificação não impõe ordem artificial e aceita categorias não vistas nas dobras de validação. Em `service`, que possui alta cardinalidade, o aumento de dimensão é controlado pela redução posterior.

### Imputação e transformação numérica

Atributos categóricos usam a moda. Atributos numéricos usam mediana e indicadores de ausência. A mediana é menos sensível a caudas e valores extremos. Depois, `StandardScaler` coloca os atributos numéricos em escala comparável.

### Redução de dimensionalidade

`TruncatedSVD(n_components=20)` reduz a matriz esparsa produzida pelo one-hot sem exigir centralização densa. Os mesmos 20 componentes são usados em todas as estratégias.

### Balanceamento

Comparamos `RandomUnderSampler` e `RandomOverSampler` contra um baseline sem balanceamento. O under-sampling reduz a classe majoritária e pode perder informação. O over-sampling replica exemplos minoritários e pode aumentar sobreajuste. Não usamos SMOTE nesta etapa porque interpolar diretamente categorias one-hot pode criar combinações categóricas fracionárias sem interpretação.

## 6. Protocolo experimental

A comparação usa cinco dobras de `StratifiedGroupKFold`, sem embaralhamento. A estratificação preserva aproximadamente a proporção das classes, enquanto o agrupamento impede que assinaturas idênticas apareçam simultaneamente no treino e na validação. Todo pré-processamento aprendido e todo balanceamento ocorrem dentro da dobra de treino.

Métricas: F1-macro, precisão e recall macro, precisão da anomalia e recall da anomalia. A acurácia não orienta a escolha devido ao desbalanceamento.

In [ ]:
results = evaluate_preprocessing_strategies(
    X,
    y,
    preparation_report,
    n_splits=5,
    n_components=20,
)
display(results.style.format({column: '{:.4f}' for column in results.columns if column.endswith(('_media', '_desvio'))}))

## 7. Exportação dos resultados

A tabela abaixo é gerada pelo experimento, evitando números copiados manualmente. Ela pode ser usada diretamente no texto da entrega.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'etapa_2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cleaning_results.to_csv(OUTPUT_DIR / 'resultados_limpeza.csv', index=False)
results.to_csv(OUTPUT_DIR / 'resultados_preprocessamento.csv', index=False)
(OUTPUT_DIR / 'diagnostico_preparacao.json').write_text(
    json.dumps(report_as_dict(preparation_report), ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('Arquivos gerados em:', OUTPUT_DIR.resolve())

## 8. Interpretação orientada pelos resultados

A estratégia indicada para a próxima etapa é a de maior F1-macro médio, desde que o desvio entre dobras e o recall da classe `anomaly` não revelem instabilidade. Se o ganho de recall vier acompanhado de grande perda de precisão, a equipe deve explicitar esse custo. O resultado não define ainda o melhor classificador; apenas seleciona uma configuração de pré-processamento para a Etapa 3.

## Declaração de uso de IA generativa

Foi utilizada uma ferramenta de IA generativa como apoio à estruturação do pipeline, à redação das justificativas e à depuração do código. As decisões metodológicas, a execução e a validação dos resultados permanecem sob responsabilidade da equipe.